# Parsiranje scenarija: *Taksista* (*Taxi Driver*, 1976)

Izdvajanje iskaza Trevisa Bikla iz scenarija za film *Taksista*. Ovo je jedan od četiri parsera urađena za master rad *Božji usamljeni ljudi: računarska analiza jezika protagonista u scenarijima Pola Šrejdera* (Univerzitet Singidunum, 2026). Svaki scenario u korpusu ima svoj parser, jer se četiri scenarija razlikuju po načinu na koji su zapisani ime lika, parentetika, dijalog i didaskalije. Postupak je opisan u potpoglavljima 5.2 i 5.3 rada.

**Izvor.** Scenario je preuzet sa sajta scriptslug.com (tačna adresa navedena je u literaturi rada) kao PDF od 114 strana. Ne sadrži oznaku verzije ni datum, a reč je o naknadnom prepisu izvornog dokumenta. Fajl nije deo repozitorijuma, jer je scenario zaštićen autorskim pravom. Za ponavljanje postupka PDF treba sačuvati kao 'Taxi-Driver-Script.pdf', u istom folderu kao
notebook.

**Izlaz.** 'travis_lines.csv', sa kolonama 'speaker', 'type', 'record_no' i 'line': 275 iskaza, od toga 239 dijalog i 36 naracije. Ovaj fajl je, zajedno sa izlazima ostala tri parsera, ulaz za notebook 'Analiza_scenarija.ipynb'.

**Pokretanje.** Redom, od prve ćelije. Potrebni paketi: 'pdfplumber' i 'pandas'.


In [1]:
import pdfplumber
import re
import pandas as pd
from collections import defaultdict
from collections import Counter

## 1. Učitavanje scenarija

PDF se čita strana po strana pomoću biblioteke pdfplumber, koja za svaku reč daje njen položaj na strani. Reči čija vertikalna pozicija, zaokružena na ceo poen, ima istu vrednost čine jedan red, a za svaki red beleže se strana, x-koordinata prve reči i tekst.


In [2]:
PDF_PATH = 'Taxi-Driver-Script.pdf'

pdf=pdfplumber.open(PDF_PATH)
all_rows = []
for pi, page in enumerate(pdf.pages):
  words = page.extract_words()
  rows = defaultdict(list)
  for w in words:
    rows[round(w['top'])].append(w) # grupisanje reči u isti red po y koordinati
  for y in sorted(rows):
    ws = sorted(rows[y], key=lambda w: w['x0'])
    x0 = round(ws[0]['x0']) # leva pozicija reda određuje tip sadržaja
    text = ' '.join(w['text'] for w in ws)
    all_rows.append({'page': pi, 'x0': x0, 'text':text})

## 2. Prepoznavanje elemenata scenarija

Uvlačenje je u ovom scenariju dosledno, pa se tip reda određuje samo po x-koordinati, kao i u parseru za *Fatalnu nesanicu*:

| x-koordinata | Tip |
|---|---|
| 245–275 | ime lika |
| 200–220 | parentetika |
| 172–190 | dijalog |
| 100–118 | didaskalija |
| ostalo | ostalo |

Parentetika koja prelazi u sledeći red ima i drugi red na istoj poziciji, pa se i on prepoznaje kao parentetika.

Od ovog pravila postoji jedan izuzetak, kojim se bavi sledeća ćelija.


In [3]:
# pozicioni pristup: uvlačenje je dosledno, osim didaskalija otkucanih kao dijalog
def classify(x0):
  if 245 <= x0 <= 275: return 'speaker' # ime lika
  if 200 <= x0 <= 220: return 'paren' # (monotone), (urgent) etc
  if 172 <= x0 <= 190: return 'dialogue'
  if 100 <= x0 <= 118: return 'action'
  return 'other'

### Didaskalije otkucane u koloni dijaloga

Na sedam mesta u scenariju didaskalija je otkucana na poziciji dijaloga, bez ikakvog tipografskog signala koji bi je odvojio od replike (primer je na 29. strani). Po poziciji bi takav pasus bio pripisan liku koji je poslednji govorio, pa je bilo potrebno sadržajno pravilo.

Pravilo traži ime lika iza kojeg, u narednih 40 znakova, sledi glagol radnje u trećem licu, kao u 'TRAVIS exits.' Ime se traži i velikim slovima, kako se likovi uvode u didaskalijama, i uobičajenim zapisom. Na kraju ćelije pravilo se proverava na četiri primera: dva pasusa didaskalije iz scenarija, koja pravilo treba da prepozna, i dve replike u kojima se javlja ime lika, koje ne sme da prepozna.


In [4]:
likovi = r"TRAVIS|BETSY|IRIS|SPORT|ANDY|WIZARD|PALANTINE|TOM|CHARLIE|OLD MAN|SECRET SERVICE MAN|POLICEMAN|Travis|Betsy|Iris|Andy"
glagoli = r"is|are|turns|exits|enters|walks|looks|smiles|stands|sits|shrugs|pauses|follows|watches|glows|sets|hands|picks|closes|reaches|pushes|rarely|motions|seen"
didaskalija = re.compile(rf"\b(?:{likovi})\b.{{0,40}}\b(?:{glagoli})\b")

print(bool(didaskalija.search("TRAVIS exits.")))
print(bool(didaskalija.search("Travis rarely smiles, but when")))
print(bool(didaskalija.search("My name is  Travis.")))
print(bool(didaskalija.search("So long, Iris. Sweet Iris.")))

True
True
False
False


### Obrazac za ime lika

Pozicija sama nije dovoljna za ime lika, pa se red na toj poziciji prihvata kao ime tek ako odgovara obrascu: velika slova, cifre, razmak, tačka, apostrof ili crtica, bez tačke na kraju.

Oznake uz ime su u ovom scenariju najraznovrsnije od sva četiri. Javljaju se, između ostalih, oblici 'TRAVIS (V.O.)(CONTD)', 'TRAVIS (V.O.; CONT'D)', 'TRAVIS (V.O.) (CONTD)' i
'TRAVIS (CONTINUED)', pa obrazac dozvoljava više sufiksa zaredom, sa zagradama ili bez njih, i više sufiksa unutar istih zagrada, odvojenih tačkom i zarezom.


In [5]:
# ime lika sa sufiksima u više oblika: (V.O.)(CONTD), (V.O.; CONT'D), (CONTINUED)...
speaker_name = re.compile(
    r"^([A-Z0-9][A-Z0-9 .'\-]{1,28}?)(?<!\.)"
    r"((?:\s*\(?(?:V\.O\.?[;]?\s*|CONT'?D|CONTINUED|O\.S\.?|O\.C\.?)+\)?)*)\s*$"
)

## 3. Parsiranje

Parser prolazi kroz redove redom i od njih sklapa iskaze. Brojevi strana se preskaču, kako ne bi prekinuli repliku koja prelazi na sledeću stranu. Ime lika otvara nov iskaz, redovi dijaloga dodaju se tekućem iskazu, a didaskalija ili bilo koji drugi red ga zatvara. Pri zatvaranju se redovi spajaju u jedan tekst, a višestruki razmaci svode na jedan. Ako uz ime stoji sufiks 'V.O.', ceo iskaz se označava kao naracija.

Parentetika ne ulazi u tekst iskaza i ne prekida ga, pa se delovi replike pre i posle nje spajaju u jedan iskaz. Blok dijaloga koji neposredno sledi parentetiku prvo se proverava pravilom iz prethodne ćelije, jer su se didaskalije otkucane kao dijalog u ovom scenariju javljale upravo na tim mestima. Ako pravilo prepozna didaskaliju, ceo blok se preskače, a tekući iskaz se zatvara.

Za razliku od *Iskušenika* i *Kockara*, ovde nema oznake '(MORE)'. Oznake '(CONT'D)' i '(CONTINUED)' uz ime označavaju nastavak replike posle didaskalije, a ne prelom strane, pa otvaraju nov iskaz, kao i u ostalim parserima, a pravilo za spajanje replike preko strane nije potrebno.


In [6]:
records = []
current, current_vo, buf = None, False, []
posle_paren = False

def flush():
  'Zatvori tekuću repliku i upiši je'
  global buf
  if current and buf:
    txt = re.sub(r'\s+', ' ', ' '.join(buf)).strip()
    if txt:
      records.append((current, current_vo, txt))
  buf = []

i = 0
while i < len(all_rows):
  r = all_rows[i]
  cls = classify(r['x0'])
  txt = r['text'].strip()
  # broj strane se preskače, da ne bi prekinuo repliku na prelomu strane
  if re.match(r"^\d+\.$", txt):
    i += 1
    continue
  if cls == 'speaker':
    m = speaker_name.match(txt)
    if m:
      flush()
      current = m.group(1).strip()
      current_vo = 'V.O.' in (m.group(2) or '')
    # red na poziciji imena koji ne odgovara obrascu nije ime lika  
    else:
      flush(); current = None
    posle_paren = False
  elif cls == 'paren':
    posle_paren = True
  elif cls == 'dialogue':
    # blok posle parentetike može biti didaskalija otkucana kao dijalog
    if posle_paren:
      j = i
      seg = []
      while j < len(all_rows) and classify(all_rows[j]['x0']) == 'dialogue':
        seg.append(all_rows[j]['text'].strip()); j += 1
      if didaskalija.search(' '.join(seg)):
        flush(); current = None
        i = j
        posle_paren = False
        continue
    posle_paren = False
    if current:
      buf.append(txt)
  else:
    flush(); current = None
    posle_paren = False
  i += 1
flush()

### Kontrola: oblici parentetike

Pregled parentetika u scenariju po učestalosti. U ispisu nema oznake naracije u obliku parentetike, jer se naracija u ovom scenariju označava sufiksom uz ime lika, a nema ni oznake '(MORE)'. Parovi poput '(casually, almost to' i 'himself)' su dva reda iste parentetike.

In [7]:
# kontrola: oblici parentetike; oznake narracije u zagradi ni (MORE) ovde nema
paren =  [r['text'] for r in all_rows if classify(r['x0']) == 'paren']
print(Counter(paren).most_common(30))

[('(a beat)', 25), ('(pause)', 11), ('(smiles)', 5), ('(to Travis)', 4), ('(laughs)', 3), ('(interrupting)', 3), ('(thinks)', 3), ('(hesitant)', 3), ('(shrugs)', 3), ('(continuing)', 2), ('(w/o inflection)', 2), ('(catching on)', 2), ('(getting up)', 2), ("(at Betsy's desk)", 2), ('(evasive)', 2), ('(slowly)', 2), ('(joking)', 2), ('(to himself)', 2), ('(nods)', 2), ('(insistent)', 2), ('(beat)', 2), ('(applause)', 2), ('(restrained anger)', 2), ('(thinks a moment)', 1), ('(pause, thin smile)', 1), ('(apologetic)', 1), ('(casually, almost to', 1), ('himself)', 1), ('(looks up at Travis)', 1), ('(rummages through his', 1)]


## 4. Izdvajanje Trevisovih iskaza i čuvanje

Svi iskazi se najpre smeštaju u tabelu i dobijaju redni broj u celom scenariju ('record_no'). Broj se dodeljuje pre izdvajanja protagoniste, pa obuhvata replike svih likova, dok didaskalije broj ne dobijaju. Zato dva Trevisova iskaza sa uzastopnim brojevima nije razdvojila replika drugog lika, nego najviše didaskalija. Na tome počiva spajanje iskaza naracije u blokove u glavnom notebooku.

Sve oznake koje počinju imenom 'TRAVIS' svode se na jedno ime, pa se Trevis vodi kao jedan govornik.

U koloni 'type' vrednost 'dialogue' označava dijalog, a 'voice_over' ono što se u radu naziva naracijom. Očekivani rezultat je 275 iskaza, od toga 239 dijalog i 36 naracije.


In [8]:
df = pd.DataFrame(records, columns=['speaker_raw', "is_vo", 'line'])
df['record_no'] = range(1, len(df) + 1) # redni broj u celom scenariju, potreban zbog kasnijeg spajanja blokova
# sve oznake koje počinju sa TRAVIS svode se na jedno ime
df["speaker"] = df['speaker_raw'].apply(lambda s: 'TRAVIS' if s. startswith("TRAVIS") else s)
travis = df[df["speaker"] == "TRAVIS"].copy().reset_index(drop=True)

travis['type'] = travis['is_vo'].apply(lambda v: "voice_over" if v else "dialogue")
travis = travis[['speaker', 'type', 'record_no', 'line']]

print('Trevisovih replika:', len(travis))
print(travis['type'].value_counts())


travis.to_csv('travis_lines.csv', index=False)

Trevisovih replika: 275
type
dialogue      239
voice_over     36
Name: count, dtype: int64
